In [ ]:
!git clone https://github.com/CryAndRRich/visenet.git

fatal: destination path 'visenet' already exists and is not an empty directory.


In [ ]:
!pip install -r visenet/requirements.txt

In [ ]:
import torch
print(torch.__version__)

2.3.1+cu121


In [ ]:
import sys
sys.path.append("/content/visenet")


In [ ]:
import pandas as pd

def preprocess_top30(df, feature_cols, top_n=30):
    """
    Chuẩn hóa data sao cho mỗi ngày có đúng top_n tickers.
    Nếu ngày nào thiếu thì fill từ ngày gần nhất (trước hoặc sau).

    df: DataFrame có cột ['ticker', 'timestamp', ... feature_cols ...]
    feature_cols: list tên các cột feature (open, high, low, close, ...).
    top_n: số lượng tickers cần giữ lại mỗi ngày.
    """
    result = []
    all_dates = sorted(df['timestamp'].unique())

    for i, date in enumerate(all_dates):
        day_df = df[df['timestamp'] == date]

        # Lấy top_n tickers (ví dụ dựa vào volume)
        top_df = day_df.nlargest(top_n, 'vol')

        # Nếu đủ top_n thì ok
        if len(top_df) == top_n:
            result.append(top_df)
        else:
            # Cần fill thêm
            missing = top_n - len(top_df)
            # Tìm ngày gần nhất có data đủ
            j = i - 1
            filled = []
            while j >= 0 and len(filled) < missing:
                prev_day = result[j]  # đã được chuẩn hóa từ trước
                # lấy ticker chưa có trong ngày hiện tại
                candidates = prev_day[~prev_day['ticker'].isin(top_df['ticker'])]
                needed = candidates.head(missing - len(filled))
                filled.append(needed)
                j -= 1
            # nếu vẫn chưa đủ thì lấy từ ngày sau
            if len(filled) < missing:
                k = i + 1
                while k < len(all_dates) and len(filled) < missing:
                    next_day = df[df['timestamp'] == all_dates[k]].nlargest(top_n, 'vol')
                    candidates = next_day[~next_day['ticker'].isin(top_df['ticker'])]
                    needed = candidates.head(missing - len(filled))
                    filled.append(needed)
                    k += 1
            # gộp lại
            filled_df = pd.concat(filled) if filled else pd.DataFrame(columns=day_df.columns)
            final_day = pd.concat([top_df, filled_df]).head(top_n)
            final_day['timestamp'] = date  # đảm bảo timestamp đúng
            result.append(final_day)

    df_out = pd.concat(result).sort_values(['timestamp', 'ticker']).reset_index(drop=True)
    df_out.to_csv("visenet/data/output/top_30_stocks_after_train_processed", index=False)
    return df_out
file_path = "visenet/data/output/top_30_stocks_after_train.csv"
df = pd.read_csv(file_path)
feature_cols = ['open','high','low','close','vol','liq','rsi','macd','cci','adx','turbulence']
data_fixed = preprocess_top30(df, feature_cols, top_n=30)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
data_fixed

,ticker,timestamp,open,high,low,close,vol,liq,rsi,macd,cci,adx,turbulence
0,ACB,20181009,7480.948202,7515.755684,7449.244572,7480.948202,0.154642,138.103964,48.737995,-10.914079,0.397042,29.789230,0.0
1,BMI,20181009,12268.596625,12268.596625,12086.647099,12112.639889,0.381410,356.472541,56.968244,287.027584,0.250250,49.882552,0.0
2,CDC,20181009,6840.937683,6840.937683,6634.885344,6634.885344,0.496036,188.390711,44.749821,-6.410665,-82.482993,7.778495,0.0
3,CHP,20181009,13222.239029,13222.239029,13164.246753,13164.246753,0.051974,109.771095,45.887039,-15.942664,6.088280,46.590709,0.0
4,DP3,20181009,19857.712843,20034.067340,19871.941940,19927.064576,0.338749,546.236621,61.798166,314.149666,87.760200,23.851662,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
51655,VIC,20250829,131100.000000,132500.000000,128100.000000,128300.000000,0.443384,5285.714286,62.433839,5657.990025,93.626106,36.691512,0.0
51656,VJC,20250829,145600.000000,146200.000000,142900.000000,144500.000000,0.501860,6378.571429,67.494846,8791.773998,68.770595,44.765681,0.0
51657,VLB,20250829,46000.000000,46100.000000,45800.000000,45950.000000,0.192969,1124.000000,45.830630,-188.053174,-74.084566,16.309929,0.0
51658,VPI,20250829,57500.000000,58300.000000,57000.000000,57000.000000,0.310553,2142.857143,58.578690,870.189572,98.553279,16.402388,0.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
def parse_actions(daily_actions):
    alerts = []
    for entry in daily_actions:
        date = entry["date"]
        stocks = entry["stocks"]
        actions = entry["action"]

        signals = []
        for s, a in zip(stocks, actions):
            if a > 0:
                signals.append(f"Gợi ý: Mua {a} cổ phiếu {s}")
            elif a < 0:
                signals.append(f"Gợi ý: Bán {abs(a)} cổ phiếu {s}")

        # Chỉ soạn mail nếu có tín hiệu
        if signals:
            text = f"Ngày {date}\n\n" + "\n".join(signals)
            alerts.append(text)

    return alerts


In [ ]:
import smtplib
from email.mime.text import MIMEText
def send_email(subject, body, to_email, from_email, app_password):
    msg = MIMEText(body, "plain", "utf-8")
    msg["Subject"] = subject
    msg["From"] = from_email
    msg["To"] = to_email

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(from_email, app_password)
        server.send_message(msg)


In [ ]:
!pip install --extra-index-url https://fiinquant.github.io/fiinquantx/simple fiinquantx
!pip install --upgrade --extra-index-url https://fiinquant.github.io/fiinquantx/simple fiinquantx

Looking in indexes: https://pypi.org/simple, https://fiinquant.github.io/fiinquantx/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.0/123.0 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 kB 4.3 MB/s eta 0:00:00
  Created wheel for fastdtw: filename=fastdtw-0.3.4-cp312-cp312-linux_x86_64.whl size=567857 sha256=3d5ebaede88beafde0e60e20f21e778d43f056838fecd96d7008e9e9f4950c94
  Stored in directory: /root/.cache/pip/wheels/ab/d0/26/b82cb0f49ae73e5e6bba4e8462fff2c9851d7bd2ec64f8891e
  Created wheel for msgpack: filename=msgpack-1.0.2-cp312-cp312-linux_x86_64.whl size=15820 sha256=d7cafe1b6940d10d85e8b421380bc211e8cc3b2986f196bebfdbc08fdc257ebd
  Stored in directory: /root/.cache/pip/wheels/67/a6/40/eda0983e595bbf3

In [ ]:
def data_split(df, start, end):
    """Tách dữ liệu thành tập huấn luyện hoặc kiểm tra dựa trên ngày tháng"""
    data = df[(df.timestamp >= start) & (df.timestamp < end)]
    data=data.sort_values(['timestamp', 'ticker'], ignore_index=True)
    data.index = data.timestamp.factorize()[0]
    return data

In [ ]:
import numpy as np
import pandas as pd

import gym
from gym import spaces
from gym.utils import seeding

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Mỗi lần giao dịch tối đa mua/bán 100 cổ phiếu
HMAX_NORMALIZE = 100
# Lượng tiền ban đầu
INITIAL_ACCOUNT_BALANCE = 1000000
# Số luợng cổ phiếu trong danh mục đầu tư
STOCK_DIM = 30
# Phí giao dịch
TRANSACTION_FEE_PERCENT = 0.001

# Chỉ số biến động: ngưỡng hợp lý 90-150
# TURBULENCE_THRESHOLD = 140
REWARD_SCALING = 1e-4

class StockEnvTrade(gym.Env):
    """Môi trường giao dịch chứng khoán cho OpenAI gym"""
    metadata = {"render.modes": ["human"]}

    def __init__(self,
                 df: pd.DataFrame,
                 day: int = 0,
                 turbulence_threshold: int = 140,
                 initial: bool = True,
                 previous_state = [],
                 model_name = "",
                 iteration = "") -> None:
        # super(StockEnv, self).__init__()

        # money = 10
        # scope = 1
        self.day = day
        self.df = df
        self.initial = initial
        self.previous_state = previous_state

        self.action_space = spaces.Box(low = -1, high = 1,shape = (STOCK_DIM,))

        # Số chiều = 181 = [Luoợng tiền hiện có] + [Giá đóng cửa điều chỉnh của 30 cổ phiếu] +
        # [Số cổ phiếu đang sở hữu của 30 cổ phiếu] + [MACD của 30 cổ phiếu] + [RSI của 30 cổ phiếu] +
        # [CCI của 30 cổ phiếu] + [ADX của 30 cổ phiếu]
        self.observation_space = spaces.Box(low=0, high=np.inf, shape=(181,))

        # Load dữ liệu
        self.data = self.df.loc[self.day,:]
        self.terminal = False
        self.turbulence_threshold = turbulence_threshold

        self.state = [INITIAL_ACCOUNT_BALANCE] + \
                      self.data.close.values.tolist() + \
                      [0] * STOCK_DIM + \
                      self.data.macd.values.tolist() + \
                      self.data.rsi.values.tolist() + \
                      self.data.cci.values.tolist() + \
                      self.data.adx.values.tolist()

        # Reward
        self.reward = 0
        self.turbulence = 0
        self.cost = 0
        self.trades = 0

        # Lưu trữ giá trị tài sản theo thời gian
        self.asset_memory = [INITIAL_ACCOUNT_BALANCE]
        self.rewards_memory = []
        # self.reset()
        self._seed()
        self.model_name = model_name
        self.iteration = iteration

    def _sell_stock(self, index, action):
        # Thực hiện hành động bán dựa trên dấu của hành động
        if self.turbulence < self.turbulence_threshold:
            if self.state[index + STOCK_DIM + 1] > 0:
                # Cập nhật số dư
                self.state[0] += self.state[index + 1] * min(abs(action), self.state[index + STOCK_DIM + 1]) * (1 - TRANSACTION_FEE_PERCENT)
                self.state[index + STOCK_DIM + 1] -= min(abs(action), self.state[index + STOCK_DIM + 1])
                self.cost += self.state[index + 1] * min(abs(action), self.state[index + STOCK_DIM + 1]) * TRANSACTION_FEE_PERCENT
                self.trades += 1
            else:
                pass
        else:
            # Nếu biến động vượt quá ngưỡng, xóa tất cả các vị trí
            if self.state[index + STOCK_DIM + 1] > 0:
                # Câp nhật số dư
                self.state[0] += self.state[index + 1] * self.state[index + STOCK_DIM + 1] * (1 - TRANSACTION_FEE_PERCENT)
                self.state[index + STOCK_DIM + 1] = 0
                self.cost += self.state[index + 1] * self.state[index + STOCK_DIM + 1] * TRANSACTION_FEE_PERCENT
                self.trades += 1
            else:
                pass

    def _buy_stock(self, index, action):
        # Thực hiện hành động mua dựa trên dấu của hành động
        if self.turbulence< self.turbulence_threshold:
            available_amount = self.state[0] // self.state[index + 1]
            # print("available_amount: {}".format(available_amount))

            # Cập nhật số dư
            self.state[0] -= self.state[index + 1] * min(available_amount, action) * (1 + TRANSACTION_FEE_PERCENT)

            self.state[index + STOCK_DIM + 1] += min(available_amount, action)

            self.cost += self.state[index + 1] * min(available_amount, action) * TRANSACTION_FEE_PERCENT
            self.trades += 1
        else:
            # Nếu biến động vượt quá ngưỡng, không mua cổ phiếu
            pass

    def step(self, actions):
        # print(self.day)
        self.terminal = self.day >= len(self.df.index.unique()) - 1
        # print(actions)

        if self.terminal:
            plt.plot(self.asset_memory, "r")
            plt.savefig("results/account_value_trade_{}_{}.png".format(self.model_name, self.iteration))
            plt.close()
            df_total_value = pd.DataFrame(self.asset_memory)
            df_total_value.to_csv("results/account_value_trade_{}_{}.csv".format(self.model_name, self.iteration))
            end_total_asset = self.state[0] + sum(np.array(self.state[1:(STOCK_DIM + 1)]) * np.array(self.state[(STOCK_DIM + 1):(STOCK_DIM * 2 + 1)]))
            print("previous_total_asset: {}".format(self.asset_memory[0]))

            print("end_total_asset: {}".format(end_total_asset))
            print("total_reward: {}".format(self.state[0] + sum(np.array(self.state[1:(STOCK_DIM + 1)]) * np.array(self.state[(STOCK_DIM + 1):(STOCK_DIM * 2 + 1)]))- self.asset_memory[0]))
            print("total_cost: ", self.cost)
            print("total trades: ", self.trades)

            df_total_value.columns = ["account_value"]
            df_total_value["daily_return"] = df_total_value.pct_change(1)
            sharpe = (4 ** 0.5) * df_total_value["daily_return"].mean() / df_total_value["daily_return"].std()
            print("Sharpe: ", sharpe)

            df_rewards = pd.DataFrame(self.rewards_memory)
            df_rewards.to_csv("results/account_rewards_trade_{}_{}.csv".format(self.model_name, self.iteration))

            # print("total asset: {}".format(self.state[0] + sum(np.array(self.state[1:29]) * np.array(self.state[29:]))))
            # with open("obs.pkl", "wb") as f:
            #     pickle.dump(self.state, f)

            return self.state, self.reward, self.terminal,{}

        else:
            # print(np.array(self.state[1:29]))

            actions = actions * HMAX_NORMALIZE
            # actions = (actions.astype(int))
            if self.turbulence >= self.turbulence_threshold:
                actions = np.array([-HMAX_NORMALIZE] * STOCK_DIM)

            begin_total_asset = self.state[0] + sum(np.array(self.state[1:(STOCK_DIM + 1)]) * np.array(self.state[(STOCK_DIM + 1):(STOCK_DIM * 2 + 1)]))
            # print("begin_total_asset: {}".format(begin_total_asset))

            argsort_actions = np.argsort(actions)

            sell_index = argsort_actions[:np.where(actions < 0)[0].shape[0]]
            buy_index = argsort_actions[::-1][:np.where(actions > 0)[0].shape[0]]

            for index in sell_index:
                # print("take sell action".format(actions[index]))
                self._sell_stock(index, actions[index])

            for index in buy_index:
                # print("take buy action: {}".format(actions[index]))
                self._buy_stock(index, actions[index])

            self.day += 1
            self.data = self.df.loc[self.day,:]
            self.turbulence = self.data["turbulence"].values[0]
            # print(self.turbulence)
            # load next state
            # print("stock_shares: {}".format(self.state[29:]))
            self.state = [self.state[0]] + \
                          self.data.close.values.tolist() + \
                          list(self.state[(STOCK_DIM + 1):(STOCK_DIM * 2 + 1)]) + \
                          self.data.macd.values.tolist() + \
                          self.data.rsi.values.tolist() + \
                          self.data.cci.values.tolist() + \
                          self.data.adx.values.tolist()

            end_total_asset = self.state[0] + sum(np.array(self.state[1:(STOCK_DIM + 1)]) * np.array(self.state[(STOCK_DIM + 1):(STOCK_DIM * 2 + 1)]))
            self.asset_memory.append(end_total_asset)
            #print("end_total_asset: {}".format(end_total_asset))

            self.reward = end_total_asset - begin_total_asset
            # print("step_reward: {}".format(self.reward))
            self.rewards_memory.append(self.reward)

            self.reward = self.reward*REWARD_SCALING

        return self.state, self.reward, self.terminal, {}

    def reset(self):
        self.day = 0
        self.data = self.df.loc[self.day, :]
        self.turbulence = 0
        self.cost = 0
        self.trades = 0
        self.terminal = False
        self.rewards_memory = []

        if self.initial or self.previous_state is None:
            # Trường hợp khởi tạo mới hoặc không có previous_state
            self.asset_memory = [INITIAL_ACCOUNT_BALANCE]
            self.state = [INITIAL_ACCOUNT_BALANCE] + \
                          self.data.close.values.tolist() + \
                          [0] * STOCK_DIM + \
                          self.data.macd.values.tolist() + \
                          self.data.rsi.values.tolist() + \
                          self.data.cci.values.tolist() + \
                          self.data.adx.values.tolist()
        else:
            try:
                previous_total_asset = (
                    self.previous_state[0] +
                    sum(
                        np.array(self.previous_state[1:(STOCK_DIM + 1)]) *
                        np.array(self.previous_state[(STOCK_DIM + 1):(STOCK_DIM * 2 + 1)])
                    )
                )
                self.asset_memory = [previous_total_asset]
                self.state = [self.previous_state[0]] + \
                              self.data.close.values.tolist() + \
                              self.previous_state[(STOCK_DIM + 1):(STOCK_DIM * 2 + 1)] + \
                              self.data.macd.values.tolist() + \
                              self.data.rsi.values.tolist() + \
                              self.data.cci.values.tolist() + \
                              self.data.adx.values.tolist()
            except Exception as e:
                print(f"[WARN] previous_state không hợp lệ, reset lại từ đầu. Chi tiết: {e}")
                self.asset_memory = [INITIAL_ACCOUNT_BALANCE]
                self.state = [INITIAL_ACCOUNT_BALANCE] + \
                              self.data.close.values.tolist() + \
                              [0] * STOCK_DIM + \
                              self.data.macd.values.tolist() + \
                              self.data.rsi.values.tolist() + \
                              self.data.cci.values.tolist() + \
                              self.data.adx.values.tolist()

        return self.state

    def render(self):
        return self.state

    def _seed(self, seed=None):
        self.np_random, seed = seeding.np_random(seed)
        return [seed]
    def save_asset_memory(self):
        return self.asset_memory


In [ ]:
# ================================================================
# Common libraries
# ================================================================
import pandas as pd
import numpy as np
import time
import gym

# ================================================================
# RL models from stable-baselines3
# ================================================================
from stable_baselines3 import A2C, PPO, TD3, SAC
from stable_baselines3.common.noise import OrnsteinUhlenbeckActionNoise
from stable_baselines3.common.vec_env import DummyVecEnv

# ================================================================
# Customized env
# ================================================================
from env.EnvMultipleStock_train import StockEnvTrain
from env.EnvMultipleStock_validation import StockEnvValidation
#from env.EnvMultipleStock_trade import StockEnvTrade
from config import config
#from preprocess.preprocessor import data_split
# ================================================================
# Training functions
# ================================================================
def train_A2C(env_train, model_name, timesteps=25000):
    """Train A2C model"""
    start = time.time()
    model = A2C("MlpPolicy", env_train, verbose=0)
    model.learn(total_timesteps=timesteps)
    end = time.time()
    model.save(f"{config.TRAINED_MODEL_DIR}/{model_name}")
    print("Training time (A2C): ", (end - start) / 60, " minutes")
    return model


def train_TD3(env_train, model_name, timesteps=10000):
    """Train TD3 model (thay cho DDPG)"""
    n_actions = env_train.action_space.shape[-1]
    action_noise = OrnsteinUhlenbeckActionNoise(
        mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions)
    )

    start = time.time()
    model = TD3("MlpPolicy", env_train, action_noise=action_noise, verbose=0)
    model.learn(total_timesteps=timesteps)
    end = time.time()
    model.save(f"{config.TRAINED_MODEL_DIR}/{model_name}")
    print("Training time (TD3): ", (end - start) / 60, " minutes")
    return model


def train_PPO(env_train, model_name, timesteps=50000):
    """Train PPO model"""
    start = time.time()
    model = PPO("MlpPolicy", env_train, ent_coef=0.005, verbose=0)
    model.learn(total_timesteps=timesteps)
    end = time.time()
    model.save(f"{config.TRAINED_MODEL_DIR}/{model_name}")
    print("Training time (PPO): ", (end - start) / 60, " minutes")
    return model

# ================================================================
# DRL prediction / validation
# ================================================================
def DRL_prediction(model, environment, test_data, test_env, test_obs):
    account_memory = []
    actions_memory = []
    daily_actions = []

    obs = test_env.reset()
    unique_trade_date = test_data.index.unique()

    for i, date in enumerate(unique_trade_date):
        action, _states = model.predict(obs)
        obs, rewards, dones, info = test_env.step(action)

        actions_memory.append(action)
        account_memory.append(test_env.env_method("save_asset_memory")[0])

        daily_actions.append({
            "date": str(date),
            "action": np.array(action).flatten().tolist(),
            "stocks": test_env.envs[0].df.ticker.unique().tolist(),
            "cash": test_env.envs[0].state[0],
            "portfolio_value": test_env.envs[0].state[0] +
                               sum(np.array(test_env.envs[0].state[1:(STOCK_DIM + 1)]) *
                                   np.array(test_env.envs[0].state[(STOCK_DIM + 1):(STOCK_DIM * 2 + 1)]))
        })

        if dones:
            break

    return account_memory, actions_memory, daily_actions



def DRL_validation(model, test_data, test_env, test_obs) -> None:
    """Validation loop"""
    for i in range(len(test_data.index.unique())):
        action, _states = model.predict(test_obs)
        test_obs, rewards, dones, info = test_env.step(action)


def get_validation_sharpe(iteration):
    """Calculate Sharpe ratio from validation results"""
    df_total_value = pd.read_csv(
        f"results/account_value_validation_{iteration}.csv", index_col=0
    )
    df_total_value.columns = ["account_value_train"]
    df_total_value["daily_return"] = df_total_value.pct_change(1)
    sharpe = (4 ** 0.5) * df_total_value["daily_return"].mean() / \
             df_total_value["daily_return"].std()
    return sharpe

# ================================================================
# Ensemble strategy
# ================================================================
def run_ensemble_strategy(df, unique_trade_date, rebalance_window, validation_window,
                          to_email, from_email, app_password) -> None:
    """
    Ensemble Strategy combining PPO, A2C and TD3
    - Train A2C, PPO, TD3 trong mỗi rebalance window
    - Chọn model tốt nhất theo Sharpe
    - Thực hiện trading với model đó
    - Log chiến lược từng ngày (daily_actions)
    - Gửi email cảnh báo cuối mỗi ngày trading
    """

    print("============Start Ensemble Strategy============")

    ppo_sharpe_list = []
    td3_sharpe_list = []
    a2c_sharpe_list = []
    model_use = []

    insample_turbulence = df[(df.timestamp < 20240101) & (df.timestamp >= 20181009)]
    insample_turbulence = insample_turbulence.drop_duplicates(subset=["timestamp"])
    insample_turbulence_threshold = np.quantile(insample_turbulence.turbulence.values, .90)

    start = time.time()
    for i in range(rebalance_window + validation_window, len(unique_trade_date), rebalance_window):
        print("============================================")

        # === turbulence threshold ===
        end_date_index = df.index[df["timestamp"] ==
                                  unique_trade_date[i - rebalance_window - validation_window]].to_list()[-1]
        end_date_index = int(end_date_index)
        start_date_index = end_date_index - validation_window * 30 + 1
        historical_turbulence = df.iloc[start_date_index:(end_date_index + 1), :]
        historical_turbulence = historical_turbulence.drop_duplicates(subset=["timestamp"])
        historical_turbulence_mean = np.mean(historical_turbulence.turbulence.values)

        if historical_turbulence_mean > insample_turbulence_threshold:
            turbulence_threshold = insample_turbulence_threshold
        else:
            turbulence_threshold = np.quantile(insample_turbulence.turbulence.values, 1)
        print("turbulence_threshold: ", turbulence_threshold)

        # === training env ===
        train = data_split(df, start=20181009,
                           end=unique_trade_date[i - rebalance_window - validation_window])
        env_train = DummyVecEnv([lambda: StockEnvTrain(train)])

        # === validation env ===
        validation = data_split(df,
                                start=unique_trade_date[i - rebalance_window - validation_window],
                                end=unique_trade_date[i - rebalance_window])
        env_val = DummyVecEnv([lambda: StockEnvValidation(validation,
                                                          turbulence_threshold=turbulence_threshold,
                                                          iteration=i)])
        obs_val = env_val.reset()

        # Train A2C
        print("======A2C Training========")
        model_a2c = train_A2C(env_train, f"A2C_30k_dow_{i}", timesteps=30000)
        DRL_validation(model_a2c, validation, env_val, obs_val)
        sharpe_a2c = get_validation_sharpe(i)
        print("A2C Sharpe Ratio: ", sharpe_a2c)

        # Train PPO
        print("======PPO Training========")
        model_ppo = train_PPO(env_train, f"PPO_100k_dow_{i}", timesteps=100000)
        DRL_validation(model_ppo, validation, env_val, obs_val)
        sharpe_ppo = get_validation_sharpe(i)
        print("PPO Sharpe Ratio: ", sharpe_ppo)

        # Train TD3
        print("======TD3 Training========")
        model_td3 = train_TD3(env_train, f"TD3_10k_dow_{i}", timesteps=10000)
        DRL_validation(model_td3, validation, env_val, obs_val)
        sharpe_td3 = get_validation_sharpe(i)
        print("TD3 Sharpe Ratio: ", sharpe_td3)

        ppo_sharpe_list.append(sharpe_ppo)
        a2c_sharpe_list.append(sharpe_a2c)
        td3_sharpe_list.append(sharpe_td3)

        # === model selection ===
        if (sharpe_ppo >= sharpe_a2c) and (sharpe_ppo >= sharpe_td3):
            model_ensemble = model_ppo
            model_use.append("PPO")
        elif (sharpe_a2c > sharpe_ppo) and (sharpe_a2c > sharpe_td3):
            model_ensemble = model_a2c
            model_use.append("A2C")
        else:
            model_ensemble = model_td3
            model_use.append("TD3")

        # === Trading ===
        print("======Trading from: ", unique_trade_date[i - rebalance_window], "to ", unique_trade_date[i])
        trade = data_split(df,
                           start=unique_trade_date[i - rebalance_window],
                           end=unique_trade_date[i])
        env_trade = DummyVecEnv([lambda: StockEnvTrade(trade)])
        obs_trade = env_trade.reset()

        account_memory, actions_memory, daily_actions = DRL_prediction(
            model_ensemble, df, trade, env_trade, obs_trade
        )

        # parse chiến lược từng ngày
        alerts = parse_actions(daily_actions)

        # gửi email mỗi ngày
        for alert in alerts:
            send_email(
                subject="Cảnh báo giao dịch từ VISENET",
                body=alert,
                to_email=to_email,
                from_email=from_email,
                app_password=app_password
            )

    end = time.time()
    print("Ensemble Strategy took: ", (end - start) / 60, " minutes")




In [ ]:
import os
import pandas as pd

def run_model() -> None:
    """Train and run ensemble trading model with email alerts."""
    os.makedirs("results", exist_ok=True)

    # =====================
    # Load and preprocess data
    # =====================
    data = data_fixed  # giả sử data_fixed đã chuẩn bị sẵn
    data['timestamp'] = data['timestamp'].astype(int)

    unique_trade_date = data[(data.timestamp > 20240101)&(data.timestamp <= 20250829)].timestamp.unique()

    print("Unique trade dates:", unique_trade_date)

    # =====================
    # Config windows
    # =====================
    rebalance_window = 63
    validation_window = 63

    # =====================
    # Email config
    # =====================
    TO_EMAIL = "ngkienn89@gmail.com"
    FROM_EMAIL = "cmfes2022@gmail.com"
    APP_PASSWORD = "pagxpirrdgkgsgti"  # app password Gmail (16 ký tự)

    # =====================
    # Run Ensemble Strategy
    # =====================
    run_ensemble_strategy(
        df=data,
        unique_trade_date=unique_trade_date,
        rebalance_window=rebalance_window,
        validation_window=validation_window,
        to_email=TO_EMAIL,
        from_email=FROM_EMAIL,
        app_password=APP_PASSWORD
    )

    print("Finished training & trading with ensemble strategy")


if __name__ == "__main__":
    run_model()


Unique trade dates: [20240102 20240103 20240104 20240105 20240108 20240109 20240110 20240111
 20240112 20240115 20240116 20240117 20240118 20240119 20240122 20240123
 20240124 20240125 20240126 20240129 20240130 20240131 20240201 20240202
 20240205 20240206 20240207 20240215 20240216 20240219 20240220 20240221
 20240222 20240223 20240226 20240227 20240228 20240229 20240301 20240304
 20240305 20240306 20240307 20240308 20240311 20240312 20240313 20240314
 20240315 20240318 20240319 20240320 20240321 20240322 20240325 20240326
 20240327 20240328 20240329 20240401 20240402 20240403 20240404 20240405
 20240408 20240409 20240410 20240411 20240412 20240415 20240416 20240417
 20240419 20240422 20240423 20240424 20240425 20240426 20240502 20240503
 20240506 20240507 20240508 20240509 20240510 20240513 20240514 20240515
 20240516 20240517 20240520 20240521 20240522 20240523 20240524 20240527
 20240528 20240529 20240530 20240531 20240603 20240604 20240605 20240606
 20240607 20240610 20240611 202

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().r

Training time (A2C):  1.5873450636863708  minutes
A2C Sharpe Ratio:  0.17989932650488732
======PPO Training========


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Training time (PPO):  4.469035164515177  minutes


...

Training time (TD3):  7.2244563857714335  minutes


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


TD3 Sharpe Ratio:  0.05498968516132469
======Trading from:  20250106 to  20250411
previous_total_asset: 1000000
end_total_asset: 1015010.3901483765
total_reward: 15010.390148376464
total_cost:  3119.597573736
total trades:  845
Sharpe:  0.05240736050227246


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


SMTPDataError: (550, b'5.4.5 Daily user sending limit exceeded. For more information on Gmail\n5.4.5 sending limits go to\n5.4.5  https://support.google.com/a/answer/166852 af79cd13be357-83634a4f965sm174226585a.71 - gsmtp')

In [ ]:
!zip -r results.zip results

  adding: results/ (stored 0%)
  adding: results/account_value_trade_ensemble_189.csv (deflated 51%)
  adding: results/account_rewards_trade_ensemble_126.csv (deflated 48%)
  adding: results/account_rewards_trade_ensemble_189.csv (deflated 49%)
  adding: results/account_value_validation_189.png (deflated 11%)
  adding: results/last_state_ensemble_189.csv (deflated 48%)
  adding: results/account_value_train.csv (deflated 52%)
  adding: results/account_value_trade_ensemble_189.png (deflated 7%)
  adding: results/account_value_train.png (deflated 7%)
  adding: results/account_value_validation_189.csv (deflated 49%)
  adding: results/account_value_validation_126.csv (deflated 50%)
  adding: results/account_value_trade_ensemble_126.png (deflated 9%)
  adding: results/account_value_validation_126.png (deflated 11%)
  adding: results/last_state_ensemble_126.csv (deflated 49%)
  adding: results/account_value_trade_ensemble_126.csv (deflated 52%)


In [ ]:
!zip -r trained_models.zip trained_models

  adding: trained_models/ (stored 0%)
  adding: trained_models/2025-09-02 15:48:18.442671/ (stored 0%)
  adding: trained_models/2025-09-02 15:48:18.442671/PPO_100k_dow_189.zip (stored 0%)
  adding: trained_models/2025-09-02 15:48:18.442671/A2C_30k_dow_189.zip (stored 0%)
  adding: trained_models/2025-09-02 15:48:18.442671/A2C_30k_dow_126.zip (stored 0%)
  adding: trained_models/2025-09-02 15:48:18.442671/TD3_10k_dow_189.zip (stored 0%)
  adding: trained_models/2025-09-02 15:48:18.442671/PPO_100k_dow_126.zip (stored 0%)
  adding: trained_models/2025-09-02 15:48:18.442671/TD3_10k_dow_126.zip (stored 0%)
